# Modelo realista · sin datos del futuro (Pablo · `realista/`)

> **Separado de `futuro/` a propósito.** Aquí el modelo solo ve el **snapshot actual y snapshots PASADOS** de la sesión,
> como funcionaría en tiempo real. Sin snapshots posteriores y sin el oráculo (etiquetas de otros controladores).
> No es `main` (la regla 1 solo permite el snapshot) ni `futuro/`. Prueba por curiosidad (21/10): ¿cuánto se puede
> mejorar sin mirar al futuro?

**Código** (reutiliza por importación funciones de `futuro/` sin modificarlas):
- `realista/features_real.py`: columnas que solo miran atrás; un `assert` falla si se cuela una que mira al futuro
  (`blk_len`, `gap_next`, `gxc_exitfl_next`).
- `realista/feat_causal.py`: bloque `causal` (`cz_*`): segundos desde el último cambio del estado que resuelve cada tarea
  (coordinación de entrada/salida, assume, nivel autorizado, rumbo, nivel de salida, zona, conflicto, ruta).
- `realista/feat_session_causal.py`: segundo paso de sesión que solo usa scores de snapshots anteriores: media en la
  racha hasta ahora, 1 y 2 snapshots antes, máximo en rachas anteriores («ya se hizo»), racha anterior y avión.
  Comprobado sin fuga: al alterar las puntuaciones futuras de una sesión no cambia ninguna fila anterior (0 de 6034).
- `realista/m_lgb_real.py`: mismos parámetros, folds y semillas que el LightGBM de futuro. Guarda en
  `realista/oof|preds|submissions` y apunta en `experiments/lgb_real_runs.csv`.

**Bloques:** `base` = columnas originales (las `a_*`, `x_*`, `k_*` del reto ya miran solo atrás) · `logic` = reglas
dentro del snapshot · `past` = `p_*`, `blk_pos`, `gap_prev` · `causal` = `cz_*` · `geo` = velocidad de acercamiento al
borde, tráfico que cruza el mismo borde y niveles (de `futuro/feat_geo.py`, sin lo que mira al futuro).

## 1 · Resultados (3 semillas, `Data/folds.csv`)

Referencia: solo el snapshot (`real_base`) = **0.6572**, el mismo nivel que el LightGBM de `main` (0.655).

| Modelo | Bloques | 1.er pase | + segundo paso causal | peor controlador (+ sesión) | frente a `real_base_ses`: sesiones / controladores suben–bajan |
|---|---|---|---|---|---|
| `real_base` | base | 0.6572 | 0.6556 | 0.568 (016) | — |
| `real_blp` | + logic + past | 0.6571 | 0.6597 | 0.591 (016) | 20/17 · 10/8 |
| `real_blpc` | + causal | 0.6645 | 0.6617 | 0.590 (016) | 26/11 · 11/6 |
| **`real_todo`** | **+ geo** | **0.6703** | **0.6718** | **0.600 (016)** | **28/9 · 15/3** |

- **Mejor modelo realista: `real_todo_ses` = 0.6718**, +1.5 sobre solo el snapshot. Mediana por sesión 0.684 (+1.2
  frente a `real_base_ses`); 15 de 18 controladores suben.
- **Lo que aporta es el pasado del avión:** cuánto hace que cambió el estado ligado a la tarea (`causal`, +0.7) y su
  movimiento reciente y el tráfico en el borde (`geo`, +0.6). `logic` y `past` no aportan.
- **El segundo paso causal apenas ayuda** (−0.3 a +0.3). Sin ver qué pasa después, el «explaining away» de la
  sesión pierde casi toda su fuerza: en futuro daba +0.7.
- **Comparación:** con futuro, el mismo LightGBM llega a 0.914 con sesión. El salto de ~0.67 a ~0.91 lo da casi entero
  saber qué hace el controlador después (siguiente instrucción, cuándo desaparece la tarea).

Envío posible (no subido): `realista/submissions/sub_real_todo_ses.csv`.


In [ ]:
# Reproducir (terminal, desde la raíz; ~8 min cada una, se pueden lanzar a la vez con --threads 7):
#   python realista/m_lgb_real.py --sesion --name real_base --feats base
#   python realista/m_lgb_real.py --sesion --name real_todo --feats base,logic,past,causal,geo --ref real_base
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "realista" else Path.cwd()
sys.path[:0] = [str(ROOT / "futuro"), str(ROOT / "cv")]
import pandas as pd, metricas
for n in ["real_base", "real_blp", "real_blpc", "real_todo"]:
    for s in ("", "_ses"):
        o = pd.read_csv(ROOT / "realista" / "oof" / f"{n}{s}.csv")
        r = pd.read_csv(ROOT / "realista" / "oof" / f"real_base{s}.csv")
        print(metricas.resumen(n + s, metricas.metricas(o, r), "real_base" + s))

## 2 · Intentos de mejora (21/10, 02:35–03:05) · resultados provisionales

Referencia: `real_todo` = 0.6703 · `real_todo_ses` = **0.6718** (3 semillas). Todo con `Data/folds.csv` y las métricas
robustas. Ninguna feature nueva mira al futuro: **prueba de fuga sobre TODAS las features realistas** (truncar cada
sesión al 40 % y al 70 % de su duración y recalcular) → 0 filas distintas de 217 k / 379 k.

| Prueba | Archivo | CV | + 2.º paso | frente a `real_todo(_ses)`: sesiones suben/bajan · controladores | Veredicto |
|---|---|---|---|---|---|
| Stack logit con los modelos de `main` (nn_tf_v2, lgb_adri, linear, types: solo snapshot) | `stack_real.py` | 0.6670 | 0.6678 | 16/18 · 6/11 | ✗ |
| Mezclas simples `real_todo` + 0.1–0.5 × (nn / lgb_adri / linear) | — | 0.668–0.6726 | — | ±0.1 | ✗ ruido |
| `hist` (h_*): tareas resueltas / nuevas deducidas del snapshot anterior, sin etiquetas | `feat_hist.py` → `real_hist` | 0.6641 | 0.6641 | 13/22 · 8/10 | ✗ |
| XGBoost rank:ndcg (GPU), mismas features | `m_gbm_real.py` → `xgb_todo` | 0.6651 | 0.6654 | 12/20 · 5/12 | ✗ solo |
| `real_todo` + 0.25 × `xgb_todo` | — | 0.6721 | — | 20/13 | ≈ ruido (+0.2) |
| CatBoost YetiRank (GPU) | `m_gbm_real.py` → `cat_todo` | 0.6398 | 0.6429 | 6/31 · 0/18 | ✗ |
| **Optuna sobre el LightGBM** (28 pruebas × 5 folds, 1 semilla) | `tune_real.py` (estudio `realista/tune/t1.log`) | **0.6684** (1 s) | pendiente | — | **prometedor** |

**Optuna (lo único que apunta a mejora).** Con 1 semilla, los parámetros de siempre dan 0.6623; la mejor prueba,
**0.6684 (+0.6)**, con mediana por sesión 0.675 y peor controlador 0.591. Las 5 mejores coinciden: `lambdarank`
(xendcg es mucho peor), **lr baja ~0.015–0.02 con ~1100–1400 árboles, `extra_trees=True`, `max_bin=63`**, subsample
~0.9, colsample ~0.75–0.9, 30–50 hojas y truncation 5–8. La mejor:
`n_estimators=1103, lr=0.0186, leaves=32, min_child=11, subsample=0.94, colsample=0.77, reg_lambda=25.6, extra_trees,
trunc=6, max_bin=63, min_gain=1e-3, cat_smooth=1.5`.

**Ojo con el ruido de semilla:** los parámetros de siempre dan 0.6623 con 1 semilla y 0.6703 con 3. El +0.6 de Optuna
hay que confirmarlo con 3 o más semillas antes de adoptarlo (la ejecución con 10 semillas, `real_todo10`, se cortó a
mitad y no dejó resultado).

**Pendiente (por orden):** (1) confirmar las 3 mejores configuraciones de Optuna con 3 y 10 semillas + 2.º paso
(`m_lgb_real.py` aún no acepta `max_bin`/`min_gain`/`cat_smooth`: hay que añadirlos o usar `tune_real.cv`);
(2) reanudar Optuna solo con lambdarank (el mismo comando continúa el estudio); (3) si se confirma, mezclarlo con
0.25 × `xgb_todo`.


In [ ]:
# Reanudar / consultar la búsqueda (terminal, desde la raíz; 3 procesos a la vez, 8 hilos cada uno):
#   python realista/tune_real.py --study t1 --trials 60 --threads 8
#   CUDA_VISIBLE_DEVICES=0 python realista/m_gbm_real.py --algo xgb --name xgb_todo --sesion --ref real_todo
#   python realista/stack_real.py real_todo nn_tf_v2 lgb_adri linear types --name stk_a --sesion --nosave
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
st = optuna.load_study(study_name="t1", storage=JournalStorage(JournalFileBackend(str(ROOT / "realista" / "tune" / "t1.log"))))
t = st.trials_dataframe().query("state == 'COMPLETE'").sort_values("value", ascending=False)
t[["number", "value", "user_attrs_ses_median", "user_attrs_min_atco"] + [c for c in t if c.startswith("params_")]].head(8)
